# LSTM Analysis SLE Labor Force Exit

## Functions

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import cross_val_predict
import random

In [ ]:
def create_sequences(data, features, max_seq_length=None):
    patients = data['PTNO'].unique()
    num_features = len(features)
    
    if not max_seq_length:
        max_seq_length = data.groupby('PTNO').size().max()
    
    X = np.zeros((len(patients), max_seq_length, num_features))
    y = np.zeros(len(patients))
    seq_lengths = np.zeros(len(patients))
    
    for i, patient in enumerate(patients):
        patient_data = data[data['PTNO'] == patient].sort_values('ASSDT')
        seq_len = len(patient_data)
        seq_lengths[i] = seq_len
        
        # Fill the sequence data
        X[i, :seq_len, :] = patient_data[features].values
        
        # Get the end_state label (last record)
        y[i] = patient_data['end_state_encoded'].iloc[-1]
    
    return X, to_categorical(y), seq_lengths



In [ ]:

def get_high_risk_patients(model, data, features, min_visits=3):
    """Optimized high-risk detection with enhanced criteria"""
    patients = data['PTNO'].unique()
    high_risk = []
    
    # Define dynamic threshold rules
    THRESHOLD_RULES = [
        {'conditions': {'SLEDAI2_I': lambda x: x >= 6, 'STERDOSE': lambda x: x >= 7.5},
         'risk_threshold': 0.4},
        {'conditions': {'ISDOSE': lambda x: x >= 100},
         'risk_threshold': 0.35},
        {'conditions': {'EMPf': lambda x: 'unemployed' in x.lower()},
         'risk_threshold': 0.3},
        {'default': True,
         'risk_threshold': 0.5}
    ]
    
    for pid in patients:
        patient_data = data[data['PTNO'] == pid].sort_values('ASSDT')
        if len(patient_data) < min_visits:
            continue
            
        risk_df = predict_risk_over_time(model, patient_data, features, X.shape[1])
        
        # Find all warning points
        warning_points = []
        for i in range(1, len(risk_df)):
            current_data = patient_data.iloc[i]
            current_risk = risk_df['Risk_Score'].iloc[i]
            
            # Apply threshold rules
            applied_threshold = 0.5
            for rule in THRESHOLD_RULES:
                if 'default' in rule:
                    applied_threshold = rule['risk_threshold']
                    break
                if all(f(current_data[k]) for k,f in rule['conditions'].items() if k in current_data):
                    applied_threshold = rule['risk_threshold']
                    break
            
            if current_risk >= applied_threshold:
                warning_points.append({
                    'date': risk_df['ASSDT'].iloc[i],
                    'risk': current_risk,
                    'context': current_data,
                    'threshold': applied_threshold
                })
        
        # Select most clinically relevant warning
        if warning_points:
            # Prioritize warnings closer to employment change
            final_status_date = patient_data['ASSDT'].iloc[-1]
            warning_points.sort(key=lambda x: abs((final_status_date - x['date']).days))
            
            best_warning = warning_points[0]
            months_prior = (final_status_date - best_warning['date']).days // 30
            
            # Only include warnings within 5 years of exit
            if months_prior <= 60:
                high_risk.append({
                    'PTNO': pid,
                    'First_High_Risk_Date': best_warning['date'],
                    'Final_Outcome': patient_data['end_state'].iloc[-1],
                    'Risk_Score': best_warning['risk'],
                    'Threshold_Used': best_warning['threshold'],
                    'SLEDAI_at_Warning': best_warning['context']['SLEDAI2_I'],
                    'Steroid_at_Warning': best_warning['context']['STERDOSE'],
                    'SDI_at_Warning': best_warning['context']['score_n'],
                    'Immunosuppressant_at_Warning': best_warning['context']['ISDOSE'],
                    'Employment_at_Warning': best_warning['context']['EMPf'],
                    'Months_Prior_to_Exit': months_prior,
                    'Warning_Type': 'Early' if months_prior > 12 else 'Imminent'
                })
    
    return pd.DataFrame(high_risk)

In [ ]:

def generate_clinical_report(high_risk_df, scaler, features):
    """Enhanced clinical report with steroid categories and ISDOSE flags"""
    # First verify what columns we actually have
    print("Available columns in high_risk_df:", high_risk_df.columns.tolist())
    
    # Create feature position mapping
    feature_pos = {feat: idx for idx, feat in enumerate(features)}
    
    # Create mapping between warning columns and feature names
    warning_to_feature = {
        'SLEDAI_at_Warning': 'SLEDAI2_I',
        'Steroid_at_Warning': 'STERDOSE',
        'SDI_at_Warning': 'score_n',
        'Immunosuppressant_at_Warning': 'ISDOSE'
    }
    
    # Create a temporary array for inverse transform
    temp_array = np.zeros((len(high_risk_df), len(features)))
    
    # Fill the temporary array with scaled values
    for warning_col, feat in warning_to_feature.items():
        if warning_col in high_risk_df.columns and feat in feature_pos:
            temp_array[:, feature_pos[feat]] = high_risk_df[warning_col]
    
    # Apply inverse transformation
    unscaled_array = scaler.inverse_transform(temp_array)
    
    # Extract the unscaled values
    high_risk_df['SLEDAI_unscaled'] = unscaled_array[:, feature_pos.get('SLEDAI2_I', 0)]
    high_risk_df['Steroid_unscaled'] = unscaled_array[:, feature_pos.get('STERDOSE', 0)]
    high_risk_df['SDI_unscaled'] = unscaled_array[:, feature_pos.get('score_n', 0)]
    high_risk_df['Immunosuppressant_unscaled'] = unscaled_array[:, feature_pos.get('ISDOSE', 0)]

    # Enhanced report with new categories
    report = {
        'Detection_Rate': f"{len(high_risk_df)}/{len(data[data['end_state'] == 'Employment Exit'])} detected",
        'Median_Warning_Time': f"{high_risk_df['Months_Prior_to_Exit'].median()} months before exit",
        'Common_Triggers': {
            'SLEDAI_Increase': f"{len(high_risk_df[high_risk_df['SLEDAI_unscaled'] >= 4])/len(high_risk_df):.0%} had SLEDAI ≥4",
            'High_Steroids': f"{len(high_risk_df[high_risk_df['Steroid_unscaled'] >= 7.5])/len(high_risk_df):.0%} on ≥7.5mg prednisone",
            'Very_High_Steroids': f"{len(high_risk_df[high_risk_df['Steroid_unscaled'] > 30])/len(high_risk_df):.0%} on >30mg prednisone",
            'High_ISDOSE': f"{len(high_risk_df[high_risk_df['Immunosuppressant_unscaled'] >= 100])/len(high_risk_df):.0%} on high ISDOSE (≥100mg)",
            'SDI_Elevated': f"{len(high_risk_df[high_risk_df['SDI_unscaled'] >= 1])/len(high_risk_df):.0%} had SDI ≥1",
            'Employment_Change': f"{len(high_risk_df[~high_risk_df['Employment_at_Warning'].str.contains('Employed', case=False)])/len(high_risk_df):.0%} showed work status decline"
        },
        'Trigger_Types': {
            'Risk_Score_Only': f"{len(high_risk_df[high_risk_df['Trigger_Type'] == 'Risk_Score'])/len(high_risk_df):.0%}" if 'Trigger_Type' in high_risk_df else "N/A",
            'Clinical_Deterioration': f"{len(high_risk_df[high_risk_df['Trigger_Type'] == 'Clinical_Deterioration'])/len(high_risk_df):.0%}" if 'Trigger_Type' in high_risk_df else "N/A"
        },
        'Median_Values': {
            'SLEDAI': high_risk_df['SLEDAI_unscaled'].median(),
            'Steroid_Dose': high_risk_df['Steroid_unscaled'].median(),
            'SDI': high_risk_df['SDI_unscaled'].median(),
            'Immunosuppressant_Dose': high_risk_df['Immunosuppressant_unscaled'].median()
        },
        'Accuracy_by_Outcome': {}
    }
    
    # Calculate accuracy by outcome if 'end_state' exists in original data
    if 'end_state' in data.columns:
        for outcome in high_risk_df['Final_Outcome'].unique():
            n_correct = len(high_risk_df[high_risk_df['Final_Outcome'] == outcome])
            n_total = len(data[data['end_state'] == outcome])
            if n_total > 0:
                report['Accuracy_by_Outcome'][outcome] = f"{n_correct/n_total:.0%}"
    
    return report, high_risk_df

In [ ]:
def plot_high_risk_trajectory_age_xaxis(pid, data, model, features, scaler):
    """Plot a patient's risk score and clinical markers with age on x-axis"""
    patient_data = data[data['PTNO'] == pid].sort_values('ASSDT')
    
    # Calculate age at each assessment
    patient_data['Age'] = (patient_data['ASSDT'] - patient_data['BIRTHDT']).dt.days / 365.25
    
    risk_df = predict_risk_over_time(model, patient_data, features, X.shape[1])
    risk_df['Age'] = (risk_df['ASSDT'] - patient_data['BIRTHDT'].iloc[0]).dt.days / 365.25
    
    # Create feature position mapping
    feature_pos = {feat: idx for idx, feat in enumerate(features)}
    
    # Prepare figure with larger width to accommodate multiple axes
    fig, ax1 = plt.subplots(figsize=(16, 8))
    
    # Plot risk score (left axis) with age on x-axis
    ax1.plot(risk_df['Age'], risk_df['Risk_Score'], 'b-', label='Exit Risk', linewidth=3)
    ax1.set_xlabel('Age (Years)', fontsize=12)
    ax1.set_ylabel('Risk Score', color='b', fontsize=12)
    ax1.axhline(y=0.7, color='r', linestyle='--', label='High-Risk Threshold')
    ax1.tick_params(axis='y', labelcolor='b')
    ax1.set_ylim(0, 1)  # Force risk score axis to start at 0 and end at 1
    
    # Create temporary array for inverse transform
    temp_array = np.zeros((len(risk_df), len(features)))
    
    # Fill with scaled values
    temp_array[:, feature_pos['SLEDAI2_I']] = risk_df['SLEDAI']
    temp_array[:, feature_pos['STERDOSE']] = risk_df['Steroid_Dose']
    temp_array[:, feature_pos['score_n']] = risk_df['SDI']
    temp_array[:, feature_pos['ISDOSE']] = risk_df['Immunosuppressant']
    
    # Apply inverse transformation
    unscaled_array = scaler.inverse_transform(temp_array)
    
    # Extract unscaled values
    risk_df['SLEDAI_unscaled'] = unscaled_array[:, feature_pos['SLEDAI2_I']]
    risk_df['Steroid_unscaled'] = unscaled_array[:, feature_pos['STERDOSE']]
    risk_df['SDI_unscaled'] = unscaled_array[:, feature_pos['score_n']]
    risk_df['Immunosuppressant_unscaled'] = unscaled_array[:, feature_pos['ISDOSE']]
    
    # Define colors and styles for each clinical marker
    clinical_markers = {
        'SLEDAI': ('g', '--'),
        'Steroid (mg)': ('m', ':'),
        'SDI': ('c', '-'),
        'Immunosuppressant (mg)': ('y', '-.')
    }
    
    # Create right axes for each clinical marker
    axes_right = {}
    offset = 60  # Pixel offset for each additional axis
    
    for i, (marker_name, (color, linestyle)) in enumerate(clinical_markers.items()):
        # Create new axis, offset from the previous one
        ax = ax1.twinx()
        ax.spines['right'].set_position(('outward', offset * i))
        
        # Plot the clinical marker
        ax.plot(risk_df['Age'], risk_df[f'{marker_name.split(" ")[0]}_unscaled'], 
               color=color, linestyle=linestyle, label=marker_name)
        
        # Style the axis and set y-lim to start at 0
        ax.set_ylabel(marker_name, color=color, fontsize=12)
        ax.tick_params(axis='y', labelcolor=color)
        
        # Get the max value for this marker to set upper bound
        max_val = risk_df[f'{marker_name.split(" ")[0]}_unscaled'].max()
        buffer = max_val * 0.25  # Add 10% buffer above max value
        ax.set_ylim(0, max_val + buffer)
        
        # Store the axis for later reference
        axes_right[marker_name] = ax
    
    # Find point where employment status changes
    status_changes = risk_df['Employment_Status'].ne(risk_df['Employment_Status'].shift())
    if status_changes.any():
        change_point = risk_df[status_changes].iloc[-1]  # Get last status change
        label = (f"Status: {change_point['Employment_Status']}\n"
                f"Age: {change_point['Age']:.1f} years\n"
                f"SLEDAI: {change_point['SLEDAI_unscaled']:.1f}\n"
                f"Steroids: {change_point['Steroid_unscaled']:.1f}mg\n"
                f"SDI: {change_point['SDI_unscaled']:.1f}\n"
                f"Immunosuppressant: {change_point['Immunosuppressant_unscaled']:.1f}mg")
        
        ax1.annotate(label, 
                    (change_point['Age'], change_point['Risk_Score']),
                    textcoords="offset points", 
                    xytext=(10,10), 
                    ha='left',
                    bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8))
    
    plt.title(f'Patient {pid} - Final Outcome: {patient_data["end_state"].iloc[-1]}', fontsize=14)
    
    # Combine legends from all axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines_right = []
    labels_right = []
    
    for ax in axes_right.values():
        lines, labels = ax.get_legend_handles_labels()
        lines_right.extend(lines)
        labels_right.extend(labels)
    
    ax1.legend(lines1 + lines_right, labels1 + labels_right, loc='upper left')
    
    plt.tight_layout()
    plt.show()

## Train Model

In [ ]:
### Train Test Split
random.seed(365)
# Create sequences for each patient


# Create sequences
X, y, seq_lengths = create_sequences(data, features)

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Build LSTM model
num_classes = y.shape[1]
model = Sequential([
    Masking(mask_value=0., input_shape=(X.shape[1], X.shape[2])),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

random.seed(365)
# Corrected model training section
from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights
y_integers = np.argmax(y_train, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_integers), y=y_integers)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

# Train the model with corrected class_weight
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    class_weight=class_weight_dict  # Use the computed weights
)

# Rest of your code remains the same...

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_acc:.4f}')

# Predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_true_classes, 
    y_pred_classes, 
    target_names=label_encoder.classes_
))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true_classes, y_pred_classes))





## Find High Patients

In [ ]:


data['BIRTHDT'] = pd.to_datetime(data['BIRTHDT'])
# Print patients with final state "Disability Benefit"
# -------------------
high_risk_df = get_high_risk_patients(model, data, features)


# Execute the analysis with age on x-axis
# -------------------
# Generate clinical report with all predictors
clinical_report, high_risk_df = generate_clinical_report(high_risk_df, scaler, features)

# Print value distributions
print("\n=== Value Distributions in High-Risk Patients (Unscaled) ===")
print("SLEDAI:\n", high_risk_df['SLEDAI_unscaled'].describe())
print("\nSteroid Doses:\n", high_risk_df['Steroid_unscaled'].describe())
print("\nSDI Scores:\n", high_risk_df['SDI_unscaled'].describe())
print("\nImmunosuppressant Doses:\n", high_risk_df['Immunosuppressant_unscaled'].describe())
print("\nEmployment Status:\n", high_risk_df['Employment_at_Warning'].value_counts())

# Print comprehensive clinical report
print("\n=== Actionable Clinical Insights ===")
print(f"Early Warning: Model flagged {len(high_risk_df)} patients at high risk")
print(f"Median lead time: {clinical_report['Median_Warning_Time']}")
print("\nKey Predictors of Labor Force Exit:")
for k, v in clinical_report['Common_Triggers'].items():
    print(f"- {v}")
print("\nMedian Values at Warning Point:")
for k, v in clinical_report['Median_Values'].items():
    print(f"- {k}: {v:.2f}")


## Visualize trajectories

In [ ]:

# Visualize example cases with age on x-axis
print("\nVisualizing High-Risk Patient Trajectories (Age on X-axis)...")
for pid in high_risk_df['PTNO'].sample(min(10, len(high_risk_df)), random_state=69):
    plot_high_risk_trajectory_age_xaxis(pid, data, model, features, scaler)
